# 1. Import libraries and data

In [1]:
import os
import numpy as np, random
import pandas as pd
import joblib

import Nitrosomonas_simulation_continuous
from scipy.stats import gaussian_kde

In [3]:
prefix = "../../"
folder_name = "3_regression/CFS-/regression_result"
fit = pd.read_csv(os.path.join(prefix, folder_name, '1_fit_results.csv'),
                  index_col='Model')
fit_3D = pd.read_csv(os.path.join(prefix, folder_name, '2_fit_results_3D.csv'),
                     index_col='Model')
fit_DR = pd.read_csv(os.path.join(prefix, folder_name, '3_fit_results_DR.csv'),
                     index_col='Field')
kde_data = np.load(os.path.join(prefix, folder_name, "4_kde_training_data.npy"))
kde_bw = float(np.load(os.path.join(prefix, folder_name, "5_kde_bandwidth.npy"))[0])
rf = joblib.load(os.path.join(prefix, folder_name, "6_rf_model.pkl"))

# 2. Parameters

In [ ]:
# reconstruct the KDE
kde_log = gaussian_kde(np.log(kde_data), bw_method=kde_bw)
  
# fitting parameters obtained from the experimental data(mean and std for each condition)
exp_params = {
    # generation time, T
    'Gtime_mu_max': fit_3D.loc['generation_time_3D', 'p0'], 
    'Gtime_r1': fit_3D.loc['generation_time_3D', 'p1'], 'Gtime_r2': fit_3D.loc['generation_time_3D', 'p2'],
    'Gtime_mu_min': fit_3D.loc['generation_time_3D', 'p3'], 
    # sigma of generation time, T
    'Gtime_sigma_min': fit_3D.loc['generation_time_3D', 'sigma_p0'], 'Gtime_sigma_max': fit_3D.loc['generation_time_3D', 'sigma_p1'],
    'Gtime_sigma_x_c': fit_3D.loc['generation_time_3D', 'sigma_p2'], 'Gtime_sigma_k': fit_3D.loc['generation_time_3D', 'sigma_p3'],
    'Gtime_min': fit_3D.loc['generation_time_3D', 'z_min'],

    # elongation rate, α
    'alpha_mu_max': fit_3D.loc['elongation_rate_3D', 'p0'], 
    'alpha_mu_r1': fit_3D.loc['elongation_rate_3D', 'p1'], 'alpha_mu_r2': fit_3D.loc['elongation_rate_3D', 'p2'], 
    'alpha_mu_x01': fit_3D.loc['elongation_rate_3D', 'p3'], 'alpha_mu_x02': fit_3D.loc['elongation_rate_3D', 'p4'],
    # sigma of elongation rate, α
    'alpha_sigma_max': fit_3D.loc['elongation_rate_3D', 'sigma_p0'],
    'alpha_sigma_r': fit_3D.loc['elongation_rate_3D', 'sigma_p1'],
    'alpha_sigma_x0': fit_3D.loc['elongation_rate_3D', 'sigma_p2'],
    'alpha_max': fit_3D.loc['elongation_rate_3D', 'z_max'], 
    
    # max cell area, A_max
    'maxAd_max': fit.loc['max_Ad', 'max_val'], # when continous simulation, max cell area observed in the experiment was used.
    # 'maxAd_mu': fit.loc['max_Ad', 'p0'], # when batch culture simulation, mean of max cell area was used.
    'Ad_sizer': fit.loc['Ad_sizer', 'p3'], 'Ad_sizer_sigma': fit.loc['Ad_sizer', 'sigma_p3'],
    
    # division ratio
    'divR_mu': fit_DR.loc['div_ratio', 'p0'], 'divR_sigma': fit_DR.loc['div_ratio', 'p1'], 
    'divR_min': fit_DR.loc['div_ratio', 'min_val'], 'divR_max': fit_DR.loc['div_ratio', 'max_val'],
    
    # KDE and RF model for initial cell properties
    'kde_log': kde_log, 'rf': rf
    } 


In [ ]:
# area per FOV
pixels_per_FOV = 1392 * 1040
area_per_pixel = (6.45 / 100)**2 # µm^2
area_per_FOV = pixels_per_FOV * area_per_pixel # µm^2
# area of microfluidic chip
area_chip = 147217883.738 # µm^2
# media volume flowing through per simulation interval(60 min)
media_vol_per_interval_uL = 8.0 * 60 # µL
media_vol_per_interval_mL = media_vol_per_interval_uL / 1000 # mL
# culture volume in one FOV
media_vol_per_interval_in_FOV = media_vol_per_interval_mL * (area_per_FOV / area_chip) # mL

fixed_params = {
    'culture_vol': media_vol_per_interval_in_FOV
}

params = {
    **exp_params,
    **fixed_params
}

# 3. Config

In [ ]:
root_path = './'
export_path = os.path.join(root_path, "simu_result_continuous")
os.makedirs(export_path, exist_ok=True)

# 4. Run simulation

In [ ]:
# initial cell number per FOV(based on the experimental data)
n_list = [45, 28, 42,
          68, 76, 69,
          57, 56, 40]
# number of simulations(based on the experimental data, 3 biological replicates x 3 FOVs)
num_simulations = 9
# total simulation time(hour)
tot_time = 2001

In [ ]:
np.random.seed(42)
random.seed(42)

for i, n_0 in enumerate(n_list):
    print(f"initial_cell_size: {n_0}")
    print(f"total_run_time: {tot_time}\n")
    print("iteration:", i)

    # update params
    params.update({
        'nbstart': n_0,
        'run_time': tot_time
    })
    
    # run simulation
    transition, cell_history = Nitrosomonas_simulation_continuous.simul_Nitrosomonas(params)

    # save results
    for subdir, data in {
        "transition": transition,
        "history": cell_history
    }.items():
        folder = os.path.join(export_path, subdir)
        os.makedirs(folder, exist_ok=True)
        data.to_json(os.path.join(folder, f"simulation_{i}.json"),
                    orient="records", lines=True)
        
    print("---------------\n")